# Train a Voicely voice on a free GPU

This notebook trains a voice from the recordings you collected in Voicely and gives you back one
`.zip` to import.

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.** When step 3 asks, choose the recordings `.zip` you downloaded from Voicely.
3. Wait. When it finishes, your browser downloads `<name>-voicely.zip`.
4. In Voicely, open the voice, choose **Import a trained model**, and pick that zip.

Keep this tab open while it runs. Free Colab sessions stop when idle and after a few hours, so for
a long dataset start with 200 epochs rather than the maximum.

Only train a voice that is your own or that you have clear permission to clone.

In [ ]:
#@title 1. Settings
VOICE_NAME = "my-voice"  #@param {type:"string"}
EPOCHS = 200  #@param {type:"slider", min:10, max:600, step:10}
BATCH_SIZE = 8  #@param {type:"slider", min:1, max:16, step:1}

import re, subprocess
VOICE_NAME = re.sub(r"[^\w\-]+", "_", VOICE_NAME).strip("_") or "my-voice"
HAS_GPU = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0
DEVICE = "cuda" if HAS_GPU else "cpu"
if HAS_GPU:
    print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip())
    print(f"GPU found. Training '{VOICE_NAME}' for {EPOCHS} epochs, batch size {BATCH_SIZE}.")
else:
    print("NO GPU. Go to Runtime > Change runtime type, pick T4 GPU, then run this cell again.")
    print("(Training on CPU works but takes many hours.)")

In [ ]:
#@title 2. Install the voice engine (about 5 minutes)
import os
FILES = {}
FILES['/content/voicely/make_requirements.py'] = '"""Turn the fork\'s requirements.txt into the set that actually installs.\n\nUsage: make_requirements.py <fork requirements.txt> <cpu|cuda>\n\nThe edits are the ones listed in RVC_ARCHITECTURE.md section 4, plus the\nworker\'s own queue client and an optional CPU-only torch build.\n"""\nimport re\nimport sys\n\nREMOVED = {"torchgen", "pathlib", "pyreadline3", "pysimplegui", "sounddevice"}\nREPLACED = {"ffmpy": "ffmpy==0.3.1"}\n# redis 4.5.5 is the newest client that accepts the fork\'s async-timeout pin.\nADDED = ["setuptools==69.5.1", "redis==4.5.5"]\n\nCPU_OVERRIDES = {\n    "torch": "torch==2.0.0+cpu",\n    "torchaudio": "torchaudio==2.0.1+cpu",\n    "onnxruntime-gpu": "onnxruntime==1.14.1",\n}\n\n\ndef main() -> None:\n    path, device = sys.argv[1], sys.argv[2]\n    if device not in ("cpu", "cuda"):\n        sys.exit(f"ENGINE_DEVICE must be cpu or cuda, got {device!r}")\n\n    out = []\n    if device == "cpu":\n        out.append("--extra-index-url https://download.pytorch.org/whl/cpu")\n\n    for raw in open(path, encoding="utf-8"):\n        line = raw.strip()\n        if not line or line.startswith("#"):\n            continue\n        name = re.split(r"[=<>!~\\[ ]", line, maxsplit=1)[0].lower()\n        if name in REMOVED:\n            continue\n        if name in REPLACED:\n            line = REPLACED[name]\n        elif device == "cpu" and name in CPU_OVERRIDES:\n            line = CPU_OVERRIDES[name]\n        out.append(line)\n\n    out.extend(ADDED)\n    print("\\n".join(out))\n\n\nif __name__ == "__main__":\n    main()\n'
FILES['/content/voicely/patch_engine.py'] = '"""Build-time fixes to the pinned RVC engine.\n\nUsage: patch_engine.py <rvc dir>\n\nEach patch asserts that its target text is present exactly once, so a changed\nupstream commit fails the image build instead of silently shipping unpatched.\n"""\nimport os\nimport sys\n\nPATCHES = [\n    # Pitch extraction for training hardcodes the GPU; fall back to CPU.\n    (\n        "extract_f0_print.py",\n        \'self.model_rmvpe = RMVPE("rmvpe.pt", is_half=False, device="cuda:0")\',\n        \'self.model_rmvpe = RMVPE("rmvpe.pt", is_half=False, \'\n        \'device="cuda:0" if torch.cuda.is_available() else "cpu")\',\n    ),\n    # With no GPU, device_count() is 0 and the trainer spawns no process at\n    # all, exiting "successfully" without training. Run one CPU process.\n    (\n        "train_nsf_sim_cache_sid_load_pretrain.py",\n        "    if torch.cuda.is_available() == False and torch.backends.mps.is_available() == True:\\n"\n        "        n_gpus = 1\\n",\n        "    if n_gpus < 1:\\n"\n        "        n_gpus = 1\\n",\n    ),\n]\n\n# Base models live on a volume; the engine looks for them by relative path.\nLINKS = {\n    "hubert_base.pt": "/models/hubert_base.pt",\n    "rmvpe.pt": "/models/rmvpe.pt",\n    "pretrained_v2/f0G40k.pth": "/models/pretrained_v2/f0G40k.pth",\n    "pretrained_v2/f0D40k.pth": "/models/pretrained_v2/f0D40k.pth",\n}\n\n\ndef main() -> None:\n    root = sys.argv[1]\n\n    for rel, old, new in PATCHES:\n        path = os.path.join(root, rel)\n        text = open(path, encoding="utf-8").read()\n        if text.count(old) != 1:\n            sys.exit(f"patch target not found exactly once in {rel}")\n        open(path, "w", encoding="utf-8").write(text.replace(old, new))\n        print(f"patched {rel}")\n\n    # The engine\'s audio loader and trainer read these files unconditionally.\n    csvdb = os.path.join(root, "csvdb")\n    os.makedirs(csvdb, exist_ok=True)\n    open(os.path.join(csvdb, "formanting.csv"), "w").write("False,1.0,1.0\\n")\n    open(os.path.join(csvdb, "stop.csv"), "w").write("False\\n")\n\n    for rel, target in LINKS.items():\n        path = os.path.join(root, rel)\n        os.makedirs(os.path.dirname(path), exist_ok=True)\n        if os.path.lexists(path):\n            os.remove(path)\n        os.symlink(target, path)\n        print(f"linked {rel} -> {target}")\n\n    for d in ("logs", "weights"):\n        os.makedirs(os.path.join(root, d), exist_ok=True)\n\n\nif __name__ == "__main__":\n    main()\n'
FILES['/content/voicely_worker/config.py'] = 'import os\nfrom pathlib import Path\n\nREDIS_URL = os.environ.get("REDIS_URL", "redis://redis:6379/0")\nDATA_DIR = Path(os.environ.get("DATA_DIR", "/data")).resolve()\nMODELS_DIR = Path(os.environ.get("MODELS_DIR", "/models")).resolve()\nRVC_DIR = Path(os.environ.get("RVC_DIR", "/opt/rvc")).resolve()\nPIPER_BIN = os.environ.get("PIPER_BIN", "/opt/piper/piper")\n\n# Queue names shared with the API (api/app/queue.py).\nQUEUE_PREFIX = "voicely:jobs:"\nEVENTS_KEY = "voicely:events"\nCANCEL_PREFIX = "voicely:cancel:"\nWORKER_PREFIX = "voicely:worker:"\n\n# Which queues this worker serves, in priority order. Run a second worker with\n# WORKER_QUEUES=train to keep long training runs from blocking conversions.\nWORKER_QUEUES = [q.strip() for q in os.environ.get("WORKER_QUEUES", "fast,train").split(",") if q.strip()]\n\n# Voice models kept in memory. Each is ~55 MB on disk but more once loaded.\nVOICE_CACHE_SIZE = int(os.environ.get("VOICE_CACHE_SIZE", "2"))\n\n# Long audio is converted in chunks of about this many seconds to bound memory.\nCHUNK_SECONDS = int(os.environ.get("CHUNK_SECONDS", "30"))\n\n# The engine\'s memory use creeps up across jobs. Past this resident size the\n# worker exits between jobs and Docker restarts it clean (0 disables).\nMAX_RSS_MB = int(os.environ.get("WORKER_MAX_RSS_MB", "5000"))\n\n# Runs the separation model twice per window for a cleaner stem (2x slower).\nMDX_DENOISE = os.environ.get("MDX_DENOISE", "0") == "1"\n\nMDX_MODEL = MODELS_DIR / "mdx" / "UVR-MDX-NET-Voc_FT.onnx"\nPIPER_DIR = MODELS_DIR / "piper"\n\n\ndef data_path(rel: str) -> Path:\n    """Resolve a queue-supplied relative path, refusing anything outside DATA_DIR."""\n    p = (DATA_DIR / rel).resolve()\n    if p != DATA_DIR and DATA_DIR not in p.parents:\n        raise ValueError(f"path escapes data dir: {rel}")\n    return p\n'
FILES['/content/voicely_worker/train.py'] = '"""Train a custom voice from a folder of recordings.\n\nDrives the fork\'s own scripts as subprocesses, in the order its GUI does:\nslice -> pitch -> HuBERT features -> fine-tune -> FAISS index. The scripts\nswallow their own errors and exit 0, so every step is checked by its output.\n"""\nimport os\nimport re\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\nfrom random import shuffle\nfrom typing import Callable\n\nimport numpy as np\n\nfrom .config import DATA_DIR, RVC_DIR\n\nSR_NAME, SR_HZ, VERSION, FEAT_DIM = "40k", 40000, "v2", 768\nEPOCH_RE = re.compile(r"====> Epoch: (\\d+)")\n\n\nclass TrainError(Exception):\n    """A training failure whose message is safe to show the user."""\n\n\nclass Canceled(Exception):\n    pass\n\n\ndef _run(cmd: list[str], log: Path, is_canceled: Callable[[], bool],\n         tick: Callable[[], None] = lambda: None) -> None:\n    """Run an engine script to completion, appending its output to `log`."""\n    with open(log, "ab") as out:\n        out.write(f"\\n$ {\' \'.join(cmd)}\\n".encode())\n        out.flush()\n        proc = subprocess.Popen(cmd, cwd=RVC_DIR, stdout=out, stderr=subprocess.STDOUT,\n                                start_new_session=True)\n        try:\n            while proc.poll() is None:\n                if is_canceled():\n                    raise Canceled()\n                tick()\n                time.sleep(2)\n        finally:\n            if proc.poll() is None:\n                # The trainer spawns children; take down the whole group.\n                os.killpg(proc.pid, signal.SIGKILL)\n                proc.wait()\n\n\ndef _tail(path: Path, limit: int = 3000) -> str:\n    if not path.is_file():\n        return ""\n    with open(path, "rb") as f:\n        f.seek(0, os.SEEK_END)\n        f.seek(max(0, f.tell() - limit))\n        return f.read().decode("utf-8", "replace")\n\n\ndef _count(d: Path, suffix: str) -> int:\n    return len([p for p in d.glob(f"*{suffix}")]) if d.is_dir() else 0\n\n\ndef build_index(feature_dir: Path, dst: Path) -> int:\n    """Build the retrieval index the same way the fork\'s train_index does."""\n    import faiss\n\n    feats = np.concatenate([np.load(p) for p in sorted(feature_dir.glob("*.npy"))], 0)\n    order = np.arange(feats.shape[0])\n    np.random.shuffle(order)\n    feats = feats[order]\n    if feats.shape[0] > 2e5:\n        from sklearn.cluster import MiniBatchKMeans\n        feats = MiniBatchKMeans(n_clusters=10000, batch_size=256 * (os.cpu_count() or 1),\n                                compute_labels=False, init="random").fit(feats).cluster_centers_\n    feats = np.ascontiguousarray(feats, dtype=np.float32)\n\n    n_ivf = max(1, min(int(16 * np.sqrt(feats.shape[0])), feats.shape[0] // 39))\n    index = faiss.index_factory(FEAT_DIM, f"IVF{n_ivf},Flat")\n    faiss.extract_index_ivf(index).nprobe = 1\n    index.train(feats)\n    for i in range(0, feats.shape[0], 8192):\n        index.add(feats[i:i + 8192])\n    faiss.write_index(index, str(dst))\n    return int(index.ntotal)\n\n\ndef train(job_id: str, dataset_dir: Path, voice_dir: Path, params: dict, *, gpu: bool,\n          emit: Callable[..., None], is_canceled: Callable[[], bool]) -> dict:\n    epochs = int(params.get("epochs", 200))\n    batch_size = int(params.get("batch_size", 8 if gpu else 4))\n    py = sys.executable\n\n    exp = "v" + job_id.replace("-", "")[:20]\n    work = DATA_DIR / "work" / "train" / exp\n    link = RVC_DIR / "logs" / exp       # the scripts only look under logs/\n    weight = RVC_DIR / "weights" / f"{exp}.pth"\n    log = work / "voicely.log"\n    exp_dir = str(link)\n\n    shutil.rmtree(work, ignore_errors=True)\n    work.mkdir(parents=True)\n    if link.is_symlink() or link.exists():\n        link.unlink()\n    link.symlink_to(work, target_is_directory=True)\n\n    try:\n        cpus = min(os.cpu_count() or 1, 8)\n\n        emit(stage="Slicing recordings", progress=0.01)\n        _run([py, "trainset_preprocess_pipeline_print.py", str(dataset_dir), str(SR_HZ),\n              str(cpus), exp_dir, "False"], log, is_canceled)\n        clips = _count(work / "0_gt_wavs", ".wav")\n        if clips < 8:\n            raise TrainError(\n                f"Only {clips} usable clips were found after removing silence. "\n                "Add more clean recordings (aim for at least 10 minutes).")\n\n        emit(stage=f"Extracting pitch ({clips} clips)", progress=0.05)\n        _run([py, "extract_f0_print.py", exp_dir, "1" if gpu else str(min(cpus, 2)), "rmvpe", "128"],\n             log, is_canceled)\n        if _count(work / "2a_f0", ".npy") == 0:\n            raise TrainError("Pitch extraction produced no output. See the training log.")\n\n        emit(stage="Extracting voice features", progress=0.12)\n        feature_cmd = [py, "extract_feature_print.py", "cuda:0" if gpu else "cpu", "1", "0"]\n        feature_cmd += ["0"] if gpu else []\n        _run(feature_cmd + [exp_dir, VERSION], log, is_canceled)\n        feature_dir = work / f"3_feature{FEAT_DIM}"\n        if _count(feature_dir, ".npy") == 0:\n            raise TrainError("Feature extraction produced no output. See the training log.")\n\n        # Clips that made it through every stage, plus the fork\'s two silent\n        # reference rows.\n        stem = lambda d, suffix: {p.name[:-len(suffix)] for p in (work / d).glob(f"*{suffix}")}  # noqa: E731\n        names = (stem("0_gt_wavs", ".wav") & stem(feature_dir.name, ".npy")\n                 & stem("2a_f0", ".wav.npy") & stem("2b-f0nsf", ".wav.npy"))\n        if len(names) < batch_size:\n            raise TrainError("Too few clips survived preprocessing to train on.")\n        rows = [f"{exp_dir}/0_gt_wavs/{n}.wav|{exp_dir}/{feature_dir.name}/{n}.npy|"\n                f"{exp_dir}/2a_f0/{n}.wav.npy|{exp_dir}/2b-f0nsf/{n}.wav.npy|0" for n in names]\n        mute = f"{RVC_DIR}/logs/mute"\n        rows += [f"{mute}/0_gt_wavs/mute{SR_NAME}.wav|{mute}/3_feature{FEAT_DIM}/mute.npy|"\n                 f"{mute}/2a_f0/mute.wav.npy|{mute}/2b-f0nsf/mute.wav.npy|0"] * 2\n        shuffle(rows)\n        (work / "filelist.txt").write_text("\\n".join(rows))\n\n        steps = -(-len(names) // batch_size)\n        log_interval = steps + 1 if steps > 1 else 1\n\n        def report():\n            done = [int(m) for m in EPOCH_RE.findall(_tail(work / "train.log", 20000))]\n            epoch = max(done, default=0)\n            emit(stage=f"Training - epoch {min(epoch + 1, epochs)} of {epochs}",\n                 progress=0.2 + 0.74 * epoch / epochs, log=_tail(work / "train.log"))\n\n        emit(stage=f"Training - epoch 1 of {epochs}", progress=0.2)\n        weight.unlink(missing_ok=True)\n        _run([py, "train_nsf_sim_cache_sid_load_pretrain.py", "-e", exp, "-sr", SR_NAME,\n              "-f0", "1", "-bs", str(batch_size), "-g", "0", "-te", str(epochs),\n              "-se", str(epochs),  # no resume support, so skip mid-run checkpoints\n              "-pg", "pretrained_v2/f0G40k.pth", "-pd", "pretrained_v2/f0D40k.pth",\n              "-l", "1", "-c", "0", "-sw", "0", "-v", VERSION, "-li", str(log_interval)],\n             log, is_canceled, tick=report)\n        if not weight.is_file():\n            raise TrainError("Training stopped before producing a voice model. "\n                             "The usual cause is running out of memory; see the training log.")\n\n        emit(stage="Building retrieval index", progress=0.95, log=_tail(work / "train.log"))\n        voice_dir.mkdir(parents=True, exist_ok=True)\n        vectors = build_index(feature_dir, voice_dir / "model.index")\n        shutil.move(str(weight), voice_dir / "model.pth")\n\n        return {"sample_rate": SR_HZ, "version": VERSION, "f0": 1, "epochs": epochs,\n                "clips": len(names), "index_vectors": vectors}\n    except TrainError:\n        # Keep the engine\'s own output next to the message for diagnosis.\n        emit(log=_tail(log, 4000))\n        raise\n    finally:\n        weight.unlink(missing_ok=True)\n        if link.is_symlink():\n            link.unlink()\n        shutil.rmtree(work, ignore_errors=True)\n'
FILES['/content/voicely_worker/__init__.py'] = ''
for path, text in FILES.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, 'w', encoding='utf-8').write(text)
print('Wrote', len(FILES), 'Voicely training files.')

import os, subprocess, sys

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

if not os.path.isdir("/content/rvc/lib"):
    sh("rm -rf /content/rvc && git init -q /content/rvc")
    sh("cd /content/rvc && git remote add origin https://github.com/Mangio621/Mangio-RVC-Fork"
       " && git fetch -q --depth 1 origin bf0ffdbc35da09b57306e429c6deda84496948a1 && git checkout -q FETCH_HEAD")

if subprocess.run("which ffmpeg", shell=True, capture_output=True).returncode != 0:
    sh("apt-get -qq update && apt-get -qq install -y ffmpeg")

# The engine needs Python 3.10 and 2023 package versions, so it gets its own
# environment instead of Colab's.
if not os.path.exists("/content/rvc-env/bin/python"):
    sh(f"{sys.executable} -m pip -q install uv")
    sh(f"{sys.executable} /content/voicely/make_requirements.py /content/rvc/requirements.txt {DEVICE} > /content/voicely/requirements.txt")
    sh("uv venv -q /content/rvc-env --python 3.10")
    sh("uv pip install -q --python /content/rvc-env/bin/python --index-strategy unsafe-best-match -r /content/voicely/requirements.txt")

os.makedirs("/models/pretrained_v2", exist_ok=True)
for name in ["hubert_base.pt", "rmvpe.pt", "pretrained_v2/f0G40k.pth", "pretrained_v2/f0D40k.pth"]:
    if not os.path.exists(f"/models/{name}"):
        sh(f"curl -fL --retry 4 -s -o /models/{name}.part https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/{name} && mv /models/{name}.part /models/{name}")

if not os.path.exists("/content/rvc/csvdb/stop.csv"):
    sh(f"{sys.executable} /content/voicely/patch_engine.py /content/rvc")
print("\nEngine installed.")

In [ ]:
#@title 3. Upload your recordings
import os, shutil, zipfile, glob

AUDIO = (".wav", ".mp3", ".flac", ".m4a", ".aac", ".ogg", ".opus", ".webm", ".mp4")
ZIP = "/content/dataset.zip"

if not os.path.exists(ZIP):
    from google.colab import files
    print("Choose the recordings .zip you downloaded from Voicely.")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file was uploaded.")
    os.rename(next(iter(uploaded)), ZIP)

shutil.rmtree("/content/dataset", ignore_errors=True)
shutil.rmtree("/content/dataset_raw", ignore_errors=True)
os.makedirs("/content/dataset")
with zipfile.ZipFile(ZIP) as z:
    z.extractall("/content/dataset_raw")
# Flatten: the trainer reads one folder and nothing but audio.
n = 0
for path in sorted(glob.glob("/content/dataset_raw/**/*", recursive=True)):
    if os.path.isfile(path) and path.lower().endswith(AUDIO):
        n += 1
        shutil.move(path, f"/content/dataset/{n:04d}{os.path.splitext(path)[1].lower()}")
shutil.rmtree("/content/dataset_raw")
if n == 0:
    raise SystemExit("The zip contains no audio files.")
print(f"{n} recordings ready.")

In [ ]:
#@title 4. Train
import subprocess
open("/content/voicely/run_training.py", "w").write('\nimport os, sys, time\nfrom pathlib import Path\nos.environ["DATA_DIR"] = "/content/voicely-data"\nos.environ["RVC_DIR"] = "/content/rvc"\nsys.path.insert(0, "/content")\nimport torch\nfrom voicely_worker import train\n\nstate = {"stage": None, "log": ""}\ndef emit(**f):\n    state["log"] = f.get("log", state["log"])\n    stage = f.get("stage")\n    if stage and stage != state["stage"]:\n        state["stage"] = stage\n        print(f"[{time.strftime(\'%H:%M:%S\')}] {round(f.get(\'progress\', 0) * 100):3d}%  {stage}", flush=True)\n\ntry:\n    meta = train.train(\n        "colab", Path("/content/dataset"), Path("/content/voice"),\n        {"epochs": int(sys.argv[1]), "batch_size": int(sys.argv[2])},\n        gpu=torch.cuda.is_available(), emit=emit, is_canceled=lambda: False)\nexcept train.TrainError as e:\n    print("\\n--- engine output ---\\n" + state["log"][-3000:])\n    sys.exit(f"TRAINING FAILED: {e}")\nprint("TRAINING DONE:", meta)\n')
print("Training starts. The percentage moves once per epoch; keep this tab open.\n")
r = subprocess.run(["/content/rvc-env/bin/python", "/content/voicely/run_training.py", str(EPOCHS), str(BATCH_SIZE)])
if r.returncode != 0:
    raise SystemExit("Training did not finish. Read the messages above.")

In [ ]:
#@title 5. Download the voice
import os, zipfile
out = f"/content/{VOICE_NAME}-voicely.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
    z.write("/content/voice/model.pth", f"{VOICE_NAME}.pth")
    z.write("/content/voice/model.index", f"added_{VOICE_NAME}.index")
print(f"Saved {out} ({os.path.getsize(out) / 1e6:.0f} MB)")
try:
    from google.colab import files
    files.download(out)
    print("Your browser is downloading it. In Voicely: open the voice > Import a trained model > choose this zip.")
except ImportError:
    pass